From the `Bamboo_setup` directory, run the following command:
```bash
jupyter notebook --no-browser --port=8888
```
Copy the URL starting with `http://localhost:8888/`.
When picking the kernel for this notebook, click **Existing Jupyter Server** and paste the URL.
Name your server 'localhost'.
From localhost, select the 'Python 3' kernel.

In [1]:
import sys
from pathlib import Path
BAMBOO_SETUP = Path.cwd()
NN_POSTPROCESSING = BAMBOO_SETUP / 'src' / 'post_processing' / 'NN'
NOTEBOOKS = NN_POSTPROCESSING / 'notebooks'
sys.path.append(str((BAMBOO_SETUP/'src').resolve()))
import pandas as pd
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', 1000)  # Set a larger width to fit the editor window
# pd.set_option('display.max_colwidth', None)  # Allow columns to be fully displayed
%load_ext autoreload

In [6]:
%autoreload 2

# Setup a data handler

In [4]:
from post_processing.NN.DataHandler import DataHandler
datahandler = DataHandler(
    workdir=Path('/eos/user/a/anunezde/Z_OUTPUT_eos/2022_even_0822/LLR_and_vars_4o5'),
    tree_name='SL_res_2b_x',
    total_inputs= NN_POSTPROCESSING / 'input/8llrs1D.txt'
)
total_df_unprep = datahandler.load_data()
total_df_unprep = datahandler.fix_any_mismatch(total_df_unprep)

Welcome to JupyROOT 6.30/02
	Loading data...


In [15]:
total_df_unprep

,event,genWeight,bjet0_pt_llr,bjets_dEta_llr,bjets_dPhi_llr,bjets_dR_llr,bjets_mbb_llr,mjj_llr,trijet_mInv_llr,trijet_pt_rat_llr,File,Process
186577,18,3.805950,-0.278,0.401,0.742,1.496,1.167,0.428,0.439,0.014,tbarWplus_dl,tW
13167,32,0.033119,-0.242,0.329,0.937,1.507,0.849,0.393,0.549,1.259,bbWW_sl,HH_bbWW
706510,36,81.103897,-0.322,-0.791,-1.330,-2.183,-2.512,-0.028,0.317,-0.649,TTbar_dl,ttbar
13168,42,0.033119,0.132,0.239,1.084,1.504,0.714,0.130,0.117,-0.098,bbWW_sl,HH_bbWW
11905,58,0.033119,0.109,0.358,0.747,1.493,1.051,-0.491,0.396,-0.674,bbWW_dl,HH_bbWW
...,...,...,...,...,...,...,...,...,...,...,...,...
30323,744899478,83745.546875,-0.288,0.327,-0.377,-0.090,-0.269,-0.151,-0.445,-0.622,Wjets_2J,WJets
30340,745108688,83745.546875,-0.223,0.343,0.757,1.520,0.263,0.193,0.558,-0.775,Wjets_2J,WJets
30318,745244862,83745.546875,-0.219,NaN,0.886,-1.550,-inf,-0.071,-1.004,0.563,Wjets_2J,WJets
30006,745282826,-83745.546875,-0.325,-0.230,0.668,-0.090,0.155,-0.419,-0.551,-0.246,Wjets_2J,WJets


# Inspect preprocessing

In [16]:
def check(df):
    nan_cells = df.isna().sum().sum()
    neg_9999_cells = (df == -9999).sum().sum()
    neg_9_cells = (df == -9).sum().sum()
    pos_9999_cells = (df == 9999).sum().sum()

    print(f"Number of cells with NaN: {nan_cells}")
    print(f"Number of cells with -9999: {neg_9999_cells}")
    print(f"Number of cells with -9: {neg_9_cells}")
    print(f"Number of cells with +9999: {pos_9999_cells}")

In [7]:
total_df_v1 = datahandler.preprocess_data(total_df_unprep, nan_replacement = -9999)
total_df_v2 = datahandler.preprocess_data(total_df_unprep, nan_replacement = -9)


Preprocessing data ...

Preprocessing data ...


# Set up your model config

In [5]:
from post_processing.NN.utils import ModelConfig
model_config = ModelConfig(
    name='multi_HH_ttbar_tW',
    type='multi',
    categorization={"HH": ["HH_bbWW"], "ttbar": ["ttbar"], "tW": ["tW"]},
    training_weight_sf={"HH_bbWW": 1.0, "ttbar": 8.0, "tW": 4.0},
    input_vars='All',
    architecture_in_yml=False,
    residual_network=False,
    # hiddenlayers=[
    #     {"type": 'Dense', "units": 16, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4}
    #     # {"type": 'Dense', "units": 16, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4},
    #     # {"type": 'Dense', "units": 16, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4}
    # ],
    # outputlayers=[
    #     {"type": 'Dense', "units": 3, "kernel_initializer": 'normal', "activation": 'softmax', "act_regularizer": {'l2': 1e-4}, "name": 'output'}
    # ],
    compiler={"optimizer": 'adam', "lr": 0.001, "loss": 'categorical_crossentropy'},
    fit={"batch_size": 1024, "epochs": 2, "validation_split": 0.25}
)

# Run DNN

### Version 1: -9999

In [12]:
from post_processing.NN.DNNModel import DNNModel
import post_processing.NN.utils as utils

DNN = DNNModel(model_config=model_config, modeldir= NOTEBOOKS/'model_custom')
model_df = DNN.set_model_df_from_total_df(total_df_v1)
X_train, X_test, Y_train, Y_test, evs_test, sw_train = DNN.Full_Splitting(model_df)
input_layer, normalized_input = DNN.input_preprocessing(X_train)

def run_custom(DNNObject, compiled_model):
    DNNObject.model = compiled_model
    DNNObject.train_model(X_train, Y_train, sw_train)
    output_df, model_metrics = DNNObject.evaluate_and_predict(X_test, Y_test, evs_test)
    utils.draw_score_distribution(DNNObject.type, output_df, DNNObject.modeldir)

Initializing model: multi_HH_ttbar_tW
	Model dataframe:
            event   genWeight  bjet0_pt_llr  bjets_dEta_llr  bjets_dPhi_llr  bjets_dR_llr  bjets_mbb_llr  mjj_llr  trijet_mInv_llr  trijet_pt_rat_llr  sample_weight  Class_HH  Class_ttbar  Class_tW
186577         18    3.805950        -0.278           0.401           0.742         1.496          1.167    0.428            0.439              0.014       3.515723         0            0         1
13167          32    0.033119        -0.242           0.329           0.937         1.507          0.849    0.393            0.549              1.259     567.997559         1            0         0
706510         36   81.103897        -0.322          -0.791          -1.330        -2.183         -2.512   -0.028            0.317             -0.649       3.159125         0            1         0
13168          42    0.033119         0.132           0.239           1.084         1.504          0.714    0.130            0.117             -0.098   

In [ ]:
def get_model1():

    